# Урок 05 - Агентский RAG


## Настройка

В этой записной книжке демонстрируется паттерн Agentic RAG (Retrieval-Augmented Generation) с использованием Microsoft Agent Framework.

**Требования:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — конечная точка вашего сервиса Azure AI Search
- `AZURE_SEARCH_API_KEY` — ключ API вашего сервиса Azure AI Search
- Развертывание Azure OpenAI, настроенное через переменные окружения
- Аутентификация в Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Что такое Agentic RAG?

Традиционный RAG следует фиксированному процессу: сначала извлечение документов, затем генерация ответа. **Agentic RAG** идет дальше, предоставляя агенту автономию решать, **когда** и **как** осуществлять поиск информации.

С помощью Agentic RAG агент может:
- **Решать**, нужен ли поиск перед ответом на вопрос
- **Выбирать**, к какому источнику данных или инструменту обращаться
- **Оценивать** полученные результаты и выполнять дальнейшие поиски, если первая попытка недостаточна
- **Объединять** информацию из нескольких шагов поиска в связный ответ

Это делает агента более гибким и точным по сравнению с фиксированным процессом «извлечение — затем генерация».


## Создание инструмента поиска

В Agentic RAG внешние источники данных оборачиваются в **инструменты**, которые агент может вызывать по требованию. Это позволяет агенту рассматривать извлечение данных как просто еще одно действие, которое он может выполнить, а не как обязательный шаг.

Ниже мы определяем базу знаний о путешествиях и предоставляем ее в виде инструмента, который агент может вызвать для получения информации о направлении.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Создание агента RAG

Теперь мы создаем агента, которому поручено **всегда извлекать информацию перед ответом**. Агент использует инструмент `search_travel_knowledge` для обоснования своих ответов на базе знаний, а не полагается на собственные обучающие данные.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итеративный поиск — шаблон Maker-Checker

Ключевым преимуществом Agentic RAG является **итеративный поиск**. Агент может выполнять несколько раундов поиска для проверки, уточнения или расширения своих первоначальных находок — аналогично рабочему процессу «maker-checker»:

1. **Шаг Maker**: агент извлекает начальную информацию и составляет ответ.
2. **Шаг Checker**: агент выполняет дополнительные поиски, чтобы проверить детали или заполнить пробелы.

Ниже агенту задаётся вопрос, требующий сравнения нескольких направлений, что побуждает его выполнить поиск несколько раз.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Сводка

В этом уроке вы научились создавать систему **Agentic RAG** с использованием Microsoft Agent Framework:

- **Agentic RAG** позволяет агентам самостоятельно решать, когда выполнять поиск информации, делая процесс извлечения динамичным, а не фиксированным.
- **Инструменты в качестве источников данных**: Внешние базы знаний (например, Azure AI Search) оборачиваются в инструменты, которые агент может вызывать.
- **Итеративный поиск**: Паттерн maker-checker позволяет агенту выполнять несколько раундов поиска — искать, проверять и уточнять — перед тем, как предоставить окончательный ответ.

В производственной среде вы замените встроенную в память `TRAVEL_KNOWLEDGE_BASE` на настоящий индекс Azure AI Search для обработки масштабного поиска по туристическим документам.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью AI-сервиса перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия обеспечить точность, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется профессиональный перевод человеком. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
